# 01. Cierre de adquisición

Este cuaderno completa la adquisición de resultados electorales y extrae los votos de Barrancabermeja por mesa y corporación.

## Cobertura


| Frente | Qué se elige | Fuente |
| --- | --- | --- |
| Alcaldía | 1 alcalde | Territoriales 2019 y 2023 |
| Concejo | 17 concejales | Territoriales 2019 y 2023 |
| Asamblea de Santander | diputados | Territoriales 2019 y 2023 |
| Gobernación de Santander | 1 gobernador | Territoriales 2019 y 2023 |
| JAL | ediles por comuna y corregimiento | Territoriales 2019 y 2023 |
| Senado y Cámara | congresistas | Congreso 2022 (y 2026, ya en el cuaderno 00) |
| Presidencia | presidente | Presidencia 2018, 2022 y 2026 |

## Archivos de entrada y salida

- Lee el estado del cuaderno 00 y los archivos de `datos/brutos/`.
- Escribe tablas en `datos/procesados/votos/` y auditorías en `docs/fuentes/auditorias/01_cierre_adquisicion/`.
- Conserva sin cambios los originales y las salidas del cuaderno 00.

## URL de descarga

El portal genera los enlaces desde un selector. Se verificaron las siguientes plantillas comprobando que cada respuesta corresponde a un archivo ZIP:

| Familia | Plantilla |
| --- | --- |
| Territoriales 2023 | `/comprimidos/MMV_TERRITORIALES2023_{DEPARTAMENTO}.zip` |
| Territoriales 2019 | `/comprimidosTwo/MMV_TERRITORIALES2019_{DEPARTAMENTO}.zip` |
| Congreso 2022 | `/congreso22/MMV_CONGRESO_2022_{DEPARTAMENTO}.zip` |
| Presidencia 2018 · 2ª vuelta | `/comprimidosTwo/MMV_PRESIDENTE_V2_2018_{DEPARTAMENTO}.zip` |
| Nacionales (presidencia y congreso 2026) | `/anexos/MMV_*.zip` |

El nombre del departamento va en mayúsculas y los espacios se reemplazan por guiones bajos, por ejemplo `SANTANDER` o `NORTE_DE_SANTANDER`.

El servidor rechaza solicitudes `HEAD`; se usa `GET`. El código electoral se obtiene de DIVIPOL y no se confunde con el código DANE (`68081`).

Las plantillas por departamento permiten descargar los archivos de los 32 departamentos.

In [1]:
# --- 1. Configuración ----------------------------------------------------
from __future__ import annotations

import json
import os
import shutil
import sys
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import pandas as pd


def raiz_proyecto(inicio: Path) -> Path:
    for carpeta in (inicio.resolve(), *inicio.resolve().parents):
        if (carpeta / "datos").is_dir() and (carpeta / "notebooks").is_dir():
            return carpeta
    raise FileNotFoundError(
        "Abre el cuaderno desde la raíz o desde notebooks/ del proyecto "
        "(las carpetas datos/ y notebooks/ deben estar presentes)."
    )


RAIZ = raiz_proyecto(Path.cwd())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.ingesta import lector_mmv as mmv  # noqa: E402  (necesita RAIZ en sys.path)

BRUTOS = RAIZ / "datos" / "brutos"
PROCESADOS = RAIZ / "datos" / "procesados"
ESTRUCTURAS = RAIZ / "docs" / "fuentes" / "estructuras"
SALIDAS = RAIZ / "docs" / "fuentes" / "auditorias" / "01_cierre_adquisicion"

FECHA_INICIO = datetime.now(timezone.utc)
ID_EJECUCION = FECHA_INICIO.strftime("%Y%m%d_%H%M%S_%f")
AUDITORIA = SALIDAS / ID_EJECUCION
AUDITORIA.mkdir(parents=True, exist_ok=False)
ESTRUCTURAS.mkdir(parents=True, exist_ok=True)

VERSION_NB = "01.1.0"
SIN_RED = os.getenv("OBSERVATORIO_SIN_RED", "0") == "1"
DESCARGAR = not SIN_RED
MAX_GIB_POR_ARCHIVO = 3
RESERVA_DISCO_GIB = 1
USER_AGENT = "ObservatorioElectoralBarrancabermeja/1.0 (investigacion; fuentes publicas)"

MUNICIPIO = "BARRANCABERMEJA"
DEPARTAMENTO = "SANTANDER"
DANE_MUNICIPIO = "68081"

# Escritura determinista: dos corridas deben producir el mismo archivo byte a byte.
OPCIONES_CSV = {"index": False, "float_format": "%.6f", "lineterminator": "\n",
                "encoding": "utf-8-sig"}


try:
    display  # type: ignore[used-before-def]
except NameError:  # el cuaderno también debe poder correrse como script
    def display(x):  # noqa: D103
        print(x.to_string(index=False) if hasattr(x, "to_string") else x)


def ahora() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def rel(p: Path) -> str:
    try:
        return Path(p).resolve().relative_to(RAIZ).as_posix()
    except ValueError:
        return Path(p).as_posix()


def escribir_csv(df: pd.DataFrame, ruta: Path) -> Path:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(ruta, **OPCIONES_CSV)
    return ruta


def escribir_json(ruta: Path, obj) -> Path:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(json.dumps(obj, ensure_ascii=False, indent=2, sort_keys=True, default=str),
                    encoding="utf-8")
    return ruta


print("Proyecto:        ", RAIZ)
print("Ejecución:       ", ID_EJECUCION)
print("Auditoría:       ", rel(AUDITORIA))
print("Descarga en red: ", DESCARGAR)
print("Lector MMV:      ", rel(Path(mmv.__file__)))

Proyecto:         D:\Users\LAURA PEREZ\Desktop\CIENCIA DE DATOS\Proy_Electoral_Bca
Ejecución:        20261003_235629_099440
Auditoría:        docs/fuentes/auditorias/01_cierre_adquisicion/20261003_235629_099440
Descarga en red:  True
Lector MMV:       src/ingesta/lector_mmv.py


## 2. Estado del cuaderno 00

Se carga el estado más reciente del cuaderno 00. Si no existe, se usa el catálogo completo.

In [2]:
AUD_00 = RAIZ / "docs" / "fuentes" / "auditorias" / "00_fuentes_y_adquisicion"
corridas_00 = sorted([d for d in AUD_00.glob("*") if (d / "estado_corpus.json").exists()]) \
    if AUD_00.is_dir() else []

estado_00: dict = {}
if corridas_00:
    ultima_00 = corridas_00[-1]
    estado_00 = json.loads((ultima_00 / "estado_corpus.json").read_text(encoding="utf-8"))
    print(f"Último estado del cuaderno 00: {ultima_00.name}")
    for clave in ("originales_centrales_completos", "listo_para_cuaderno_01",
                  "pendientes_centrales", "pendientes_ampliacion"):
        if clave in estado_00:
            print(f"  {clave}: {estado_00[clave]}")
else:
    print("No hay auditorías del cuaderno 00. Se trabaja con la lista completa de fuentes.")

pendientes_00 = list(estado_00.get("pendientes_centrales", []))
codigo_00 = estado_00.get("codigo_electoral_2026_1v", {})
if codigo_00:
    print(f"\nCódigo electoral observado en 2026 1V: depto {codigo_00.get('departamento')} · "
          f"municipio {codigo_00.get('municipio')}")
    print("Este cuaderno NO lo reutiliza: lo vuelve a descubrir en el DIVIPOL de cada paquete.")

Último estado del cuaderno 00: 20261003_164243_810157
  originales_centrales_completos: False
  listo_para_cuaderno_01: False
  pendientes_centrales: ['REG_2023_TERR', 'REG_2019_TERR', 'REG_2022_CONGRESO', 'REG_CENSO_HIST', 'REG_CANDIDATURAS_2019', 'REG_CANDIDATURAS_2023']
  pendientes_ampliacion: ['REG_2015_TERR', 'REG_2018_CONGRESO', 'DANE_CNPV_2018', 'GEO_COMUNAS']

Código electoral observado en 2026 1V: depto 27 · municipio 019
Este cuaderno NO lo reutiliza: lo vuelve a descubrir en el DIVIPOL de cada paquete.


## 3. Catálogo de fuentes

`obligatoria` identifica las fuentes requeridas para cerrar esta etapa.

In [3]:
PORTAL = "https://observatorio.registraduria.gov.co"
PAGINA_HIST = f"{PORTAL}/views/electoral/historicos-resultados.php"

PLANTILLAS = {
    "territoriales_2023": f"{PORTAL}/comprimidos/MMV_TERRITORIALES2023_{{dep}}.zip",
    "territoriales_2019": f"{PORTAL}/comprimidosTwo/MMV_TERRITORIALES2019_{{dep}}.zip",
    "congreso_2022": f"{PORTAL}/congreso22/MMV_CONGRESO_2022_{{dep}}.zip",
    "presidencia_2018_2v": f"{PORTAL}/comprimidosTwo/MMV_PRESIDENTE_V2_2018_{{dep}}.zip",
}


def clave_departamento(nombre: str) -> str:
    """SANTANDER, NORTE_DE_SANTANDER, LA_GUAJIRA: mayúsculas y guion bajo."""
    return mmv.canon(nombre).replace(" ", "_")


DEP = clave_departamento(DEPARTAMENTO)

FUENTES = [
    {"id_fuente": "REG_2023_TERR", "familia": "territorial", "anio": "2023",
     "titulo": "Territoriales 2023 · Santander · Gobernación, Asamblea, Alcaldía, Concejo y JAL",
     "url": PLANTILLAS["territoriales_2023"].format(dep=DEP),
     "destino": BRUTOS / "registraduria" / "2023" / f"MMV_TERRITORIALES2023_{DEP}.zip",
     "obligatoria": True},
    {"id_fuente": "REG_2019_TERR", "familia": "territorial", "anio": "2019",
     "titulo": "Territoriales 2019 · Santander · Gobernación, Asamblea, Alcaldía, Concejo y JAL",
     "url": PLANTILLAS["territoriales_2019"].format(dep=DEP),
     "destino": BRUTOS / "registraduria" / "2019" / f"MMV_TERRITORIALES2019_{DEP}.zip",
     "obligatoria": True},
    {"id_fuente": "REG_2022_CONGRESO", "familia": "congreso", "anio": "2022",
     "titulo": "Congreso 2022 · Santander · Senado y Cámara",
     "url": PLANTILLAS["congreso_2022"].format(dep=DEP),
     "destino": BRUTOS / "congreso" / "2022" / f"MMV_CONGRESO_2022_{DEP}.zip",
     "obligatoria": True},
    {"id_fuente": "REG_2018_PRES_2V_DEP", "familia": "presidencia", "anio": "2018",
     "titulo": "Presidencia 2018 · 2ª vuelta · Santander (paquete departamental)",
     "url": PLANTILLAS["presidencia_2018_2v"].format(dep=DEP),
     "destino": BRUTOS / "registraduria" / "2018" / f"MMV_PRESIDENTE_V2_2018_{DEP}.zip",
     "obligatoria": False},
]

catalogo = pd.DataFrame([{**f, "destino": rel(f["destino"])} for f in FUENTES])
display(catalogo[["id_fuente", "titulo", "anio", "obligatoria", "url"]])
escribir_csv(catalogo, AUDITORIA / "catalogo_fuentes_resueltas.csv")
escribir_json(AUDITORIA / "plantillas_url.json", {
    "portal": PORTAL, "pagina": PAGINA_HIST, "plantillas": PLANTILLAS,
    "formato_departamento": "MAYÚSCULAS con guion bajo (NORTE_DE_SANTANDER, LA_GUAJIRA)",
    "nota_head": "El servidor rechaza HEAD y responde 500 con HTML. Pedir siempre con GET.",
    "verificacion": "Cada plantilla se comprobó leyendo los primeros bytes de la respuesta "
                    "y confirmando la firma PK de un archivo ZIP.",
})

,id_fuente,titulo,anio,obligatoria,url
0,REG_2023_TERR,"Territoriales 2023 · Santander · Gobernación, ...",2023,True,https://observatorio.registraduria.gov.co/comp...
1,REG_2019_TERR,"Territoriales 2019 · Santander · Gobernación, ...",2019,True,https://observatorio.registraduria.gov.co/comp...
2,REG_2022_CONGRESO,Congreso 2022 · Santander · Senado y Cámara,2022,True,https://observatorio.registraduria.gov.co/cong...
3,REG_2018_PRES_2V_DEP,Presidencia 2018 · 2ª vuelta · Santander (paqu...,2018,False,https://observatorio.registraduria.gov.co/comp...


WindowsPath('D:/Users/LAURA PEREZ/Desktop/CIENCIA DE DATOS/Proy_Electoral_Bca/docs/fuentes/auditorias/01_cierre_adquisicion/20261003_235629_099440/plantillas_url.json')

## 4. Descarga de originales

Se omiten los ZIP válidos ya existentes. Las descargas se transmiten por bloques a un archivo temporal y se renombran al completarse. Se rechazan archivos que excedan el tamaño máximo.

In [4]:
def es_zip_legible(ruta: Path) -> bool:
    try:
        with zipfile.ZipFile(ruta) as z:
            z.namelist()
        return True
    except (zipfile.BadZipFile, OSError):
        return False


def adquirir(fuente: dict) -> dict:
    destino: Path = fuente["destino"]
    reg = {"id_fuente": fuente["id_fuente"], "url": fuente["url"],
           "ruta_relativa": rel(destino), "momento": ahora(), "detalle": ""}

    if destino.exists() and es_zip_legible(destino):
        reg.update(estado="existente_validado", bytes=destino.stat().st_size,
                   sha256=mmv.sha256_archivo(destino))
        return reg
    if not DESCARGAR:
        reg.update(estado="omitida_sin_red", bytes=0, sha256=None,
                   detalle="OBSERVATORIO_SIN_RED=1")
        return reg

    destino.parent.mkdir(parents=True, exist_ok=True)
    libres = shutil.disk_usage(destino.parent).free / (1 << 30)
    if libres < MAX_GIB_POR_ARCHIVO + RESERVA_DISCO_GIB:
        reg.update(estado="sin_espacio_en_disco", bytes=0, sha256=None,
                   detalle=f"libres {libres:.1f} GiB")
        return reg

    parcial = destino.with_suffix(destino.suffix + ".parcial")
    peticion = urllib.request.Request(fuente["url"], headers={"User-Agent": USER_AGENT})
    try:
        with urllib.request.urlopen(peticion, timeout=180) as r:
            declarado = int(r.headers.get("Content-Length") or 0)
            if declarado > MAX_GIB_POR_ARCHIVO * (1 << 30):
                reg.update(estado="rechazada_por_tamano", bytes=declarado, sha256=None,
                           detalle=f"{declarado / (1 << 30):.2f} GiB declarados")
                return reg
            escritos = 0
            with open(parcial, "wb") as f:
                while True:
                    trozo = r.read(1 << 20)
                    if not trozo:
                        break
                    if escritos == 0 and not trozo.startswith(b"PK"):
                        reg.update(estado="respuesta_no_es_zip", bytes=0, sha256=None,
                                   detalle=trozo[:80].decode("latin-1", "replace"))
                        parcial.unlink(missing_ok=True)
                        return reg
                    f.write(trozo)
                    escritos += len(trozo)
                    if escritos > MAX_GIB_POR_ARCHIVO * (1 << 30):
                        raise RuntimeError("la descarga superó el límite de tamaño")
        if not es_zip_legible(parcial):
            parcial.unlink(missing_ok=True)
            reg.update(estado="zip_corrupto", bytes=0, sha256=None,
                       detalle="la descarga terminó pero el ZIP no se puede abrir")
            return reg
        parcial.replace(destino)
        reg.update(estado="descargado_validado", bytes=destino.stat().st_size,
                   sha256=mmv.sha256_archivo(destino))
    except Exception as e:  # noqa: BLE001
        parcial.unlink(missing_ok=True)
        reg.update(estado="fallo_descarga", bytes=0, sha256=None,
                   detalle=f"{type(e).__name__}: {e}")
    return reg


adquisiciones = [adquirir(f) for f in FUENTES]
tabla_adq = pd.DataFrame(adquisiciones)
tabla_adq["MB"] = (tabla_adq["bytes"].fillna(0) / 1_048_576).round(1)
display(tabla_adq[["id_fuente", "estado", "MB", "detalle"]])
escribir_csv(tabla_adq, AUDITORIA / "adquisiciones.csv")

OK = {"existente_validado", "descargado_validado", "manual_validado"}
disponibles = [a for a in adquisiciones if a["estado"] in OK]
print(f"\nPaquetes listos para procesar: {len(disponibles)} de {len(FUENTES)}")
for a in adquisiciones:
    if a["estado"] not in OK:
        print(f"  pendiente · {a['id_fuente']}: {a['estado']} {a['detalle']}")

,id_fuente,estado,MB,detalle
0,REG_2023_TERR,existente_validado,13.9,
1,REG_2019_TERR,existente_validado,11.6,
2,REG_2022_CONGRESO,existente_validado,5.9,
3,REG_2018_PRES_2V_DEP,fallo_descarga,0.0,PermissionError: [WinError 32] El proceso no t...



Paquetes listos para procesar: 3 de 4
  pendiente · REG_2018_PRES_2V_DEP: fallo_descarga PermissionError: [WinError 32] El proceso no tiene acceso al archivo porque está siendo utilizado por otro proceso: 'D:\\Users\\LAURA PEREZ\\Desktop\\CIENCIA DE DATOS\\Proy_Electoral_Bca\\datos\\brutos\\registraduria\\2018\\MMV_PRESIDENTE_V2_2018_SANTANDER.zip.parcial'


## 5. Inventario y estructuras

Se inventarían los archivos de cada paquete y se guarda el PDF de estructuras en `docs/fuentes/estructuras/` junto con su huella SHA-256.

In [5]:
inventario, estructuras = [], []
for a in disponibles:
    ruta = RAIZ / a["ruta_relativa"]
    with zipfile.ZipFile(ruta) as z:
        for i in z.infolist():
            if i.is_dir():
                continue
            inventario.append({
                "id_fuente": a["id_fuente"], "miembro": i.filename,
                "nombre": PurePosixPath(i.filename).name,
                "extension": PurePosixPath(i.filename).suffix.lower(),
                "bytes": i.file_size, "comprimido": i.compress_size,
                "crc32": f"{i.CRC:08x}",
            })
        for pdf in mmv.miembros_por_patron(z, r"\.pdf$"):
            datos = z.read(pdf)
            nombre = f"{a['id_fuente']}__{PurePosixPath(pdf).name}"
            (ESTRUCTURAS / nombre).write_bytes(datos)
            estructuras.append({"id_fuente": a["id_fuente"], "miembro": pdf,
                                "copia": rel(ESTRUCTURAS / nombre),
                                "sha256": mmv.sha256_bytes(datos), "bytes": len(datos)})

tabla_inv = pd.DataFrame(inventario)
if len(tabla_inv):
    display(tabla_inv.groupby(["id_fuente", "extension"])
            .agg(archivos=("miembro", "size"),
                 MB=("bytes", lambda s: round(s.sum() / 1_048_576, 1)))
            .reset_index())
    escribir_csv(tabla_inv, AUDITORIA / "inventario_miembros.csv")
if estructuras:
    display(pd.DataFrame(estructuras)[["id_fuente", "copia", "bytes"]])
    escribir_csv(pd.DataFrame(estructuras), AUDITORIA / "pdf_estructuras.csv")
    print("\nEl PDF de estructuras de cada paquete quedó guardado. Es la referencia a citar\n"
          "cuando haya que justificar la interpretación de un campo.")

,id_fuente,extension,archivos,MB
0,REG_2019_TERR,.csv,1,160.6
1,REG_2019_TERR,.txt,1,0.0
2,REG_2022_CONGRESO,.csv,1,194.9
3,REG_2022_CONGRESO,.txt,1,0.0
4,REG_2023_TERR,.csv,1,183.8
5,REG_2023_TERR,.txt,1,0.0


## 6. Formatos de los paquetes

Los paquetes MMV usan dos formatos:

- **Consolidado:** un CSV con los votos y las etiquetas de mesa, corporación, partido y candidatura (territoriales 2019 y 2023, Congreso 2022).
- **Con catálogos:** DIVIPOL, candidatos y partidos separados, además de archivos de preconteo y escrutinio (Presidencia y Congreso 2026).

El formato se detecta a partir de los archivos del ZIP y se registra en la auditoría.

In [6]:
formatos = []
for a in disponibles:
    det = mmv.detectar_formato(RAIZ / a["ruta_relativa"])
    formatos.append({"id_fuente": a["id_fuente"], "formato": det["formato"],
                     "miembro_datos": det["miembro"], "razon": det["razon"]})
    a["formato"] = det["formato"]
    a["miembro_datos"] = det["miembro"]

tabla_formatos = pd.DataFrame(formatos)
display(tabla_formatos[["id_fuente", "formato", "razon"]])
escribir_csv(tabla_formatos.astype(str), AUDITORIA / "formatos_detectados.csv")

desconocidos = tabla_formatos[tabla_formatos["formato"] == "desconocido"]
if len(desconocidos):
    print("\nPaquetes cuya arquitectura no se reconoce. No se interpretan; se registran "
          "sus miembros para revisarlos:")
    for a in disponibles:
        if a.get("formato") == "desconocido":
            det = mmv.detectar_formato(RAIZ / a["ruta_relativa"])
            escribir_json(AUDITORIA / f"formato_desconocido_{a['id_fuente'].lower()}.json", det)
            print(f"  {a['id_fuente']}: {det['miembros'][:10]}")

,id_fuente,formato,razon
0,REG_2023_TERR,csv_consolidado,un único CSV de datos: MMV_2023_27_SANTANDER.csv
1,REG_2019_TERR,csv_consolidado,un único CSV de datos: MMV_2019_27_SANTANDER.csv
2,REG_2022_CONGRESO,csv_consolidado,un único CSV de datos: MMV_2022_27_SANTANDER.csv


## 7. Verificación de archivos

Se compara el SHA-256 de cada archivo con el valor publicado en el manifiesto. Si no coincide, se detiene su procesamiento.

In [7]:
huellas = []
for a in disponibles:
    if a.get("formato") != "csv_consolidado":
        continue
    h = mmv.verificar_huella_publicada(RAIZ / a["ruta_relativa"], a["miembro_datos"])
    huellas.append({"id_fuente": a["id_fuente"],
                    "archivo": PurePosixPath(a["miembro_datos"]).name, **h})

if huellas:
    tabla_huellas = pd.DataFrame(huellas)
    display(tabla_huellas[["id_fuente", "archivo", "estado"]])
    escribir_csv(tabla_huellas.astype(str), AUDITORIA / "verificacion_huellas.csv")
    alteradas = tabla_huellas[tabla_huellas["estado"] == "NO_COINCIDE"]
    if len(alteradas):
        raise AssertionError(
            "El SHA-256 de estos archivos no coincide con el manifiesto oficial del "
            f"paquete: {alteradas['id_fuente'].tolist()}. Vuelve a descargarlos antes de "
            "seguir; no se analiza un archivo que no es el publicado."
        )
    print("\nTodas las huellas coinciden con las que publica la Registraduría.")

,id_fuente,archivo,estado
0,REG_2023_TERR,MMV_2023_27_SANTANDER.csv,coincide
1,REG_2019_TERR,MMV_2019_27_SANTANDER.csv,coincide
2,REG_2022_CONGRESO,MMV_2022_27_SANTANDER.csv,coincide



Todas las huellas coinciden con las que publica la Registraduría.


## 8. Extracción de votos

Los CSV departamentales se leen por bloques y se conservan las filas de Barrancabermeja.

El código electoral municipal se identifica a partir del nombre. Si el nombre corresponde a más de un código, el procesamiento se detiene.

Las filas se clasifican sin eliminar registros:

| `clase_opcion` | Qué es |
| --- | --- |
| `candidatura` | Voto por una persona de una lista |
| `voto_de_lista_no_preferente` | Voto a la lista sin marcar candidato |
| `opcion_especial` | Voto en blanco, nulo o no marcado (códigos 996, 997, 998) |
| `agregado_sin_candidatura` | Fila de totales que el propio archivo trae |

Se conservan todas las filas, incluidos los votos en blanco, nulos y no marcados.

In [8]:
votos, resumenes, incidencias = {}, [], []
for a in disponibles:
    idf = a["id_fuente"]
    if a.get("formato") != "csv_consolidado":
        continue
    try:
        d, r = mmv.leer_mmv_csv(RAIZ / a["ruta_relativa"], a["miembro_datos"],
                                MUNICIPIO, DEPARTAMENTO)
        anio = next(f["anio"] for f in FUENTES if f["id_fuente"] == idf)
        familia = next(f["familia"] for f in FUENTES if f["id_fuente"] == idf)
        d["id_fuente"], d["anio"], d["familia"] = idf, anio, familia
        votos[idf] = d
        resumenes.append({"id_fuente": idf, "anio": anio, **r})
    except mmv.EstructuraDesconocida as e:
        escribir_json(AUDITORIA / f"estructura_pendiente_{idf.lower()}.json",
                      {"id_fuente": idf, "mensaje": str(e), **e.diagnostico})
        incidencias.append({"id_fuente": idf, "problema": "estructura no reconocida",
                            "detalle": str(e)})
    except Exception as e:  # noqa: BLE001
        incidencias.append({"id_fuente": idf, "problema": type(e).__name__,
                            "detalle": str(e)})

if resumenes:
    tr = pd.DataFrame(resumenes)
    display(tr[["id_fuente", "anio", "codificacion", "columnas", "cod_municipio",
                "registros_departamento", "registros_municipio", "puestos",
                "codigos_de_puesto", "mesas", "votos"]])
    escribir_csv(tr.astype(str), AUDITORIA / "extraccion_por_paquete.csv")
    for r in resumenes:
        print(f"\n{r['id_fuente']} ({r['anio']}) · corporaciones: {', '.join(r['corporaciones'])}")
        print(f"  clases de opción: {r['clases_opcion']}")
        if r["comunas_con_codigo_pegado"]:
            print(f"  {r['comunas_con_codigo_pegado']:,} filas traen el código de comuna "
                  "pegado al nombre; se conserva el original y se agrega comuna_normalizada.")
        if r["puestos"] != r["codigos_de_puesto"]:
            print(f"  ATENCIÓN: {r['codigos_de_puesto']} códigos de puesto para "
                  f"{r['puestos']} puestos reales. Hay que agrupar por id_puesto, "
                  "nunca por cod_puesto.")

if incidencias:
    print("\nIncidencias:")
    display(pd.DataFrame(incidencias))
    escribir_csv(pd.DataFrame(incidencias), AUDITORIA / "incidencias.csv")

,id_fuente,anio,codificacion,columnas,cod_municipio,registros_departamento,registros_municipio,puestos,codigos_de_puesto,mesas,votos
0,REG_2023_TERR,2023,utf-8,18,019,951067,138142,67,67,593,649758
1,REG_2019_TERR,2019,utf-8,18,019,847729,116630,63,63,537,600674
2,REG_2022_CONGRESO,2022,utf-16,21,019,446213,48207,61,56,547,202035



REG_2023_TERR (2023) · corporaciones: ALCALDE, ASAMBLEA, CONCEJO, GOBERNADOR, JAL
  clases de opción: {'candidatura': 116532, 'voto_de_lista_no_preferente': 13099, 'opcion_especial': 8511}

REG_2019_TERR (2019) · corporaciones: ALCALDE, ASAMBLEA, CONCEJO, GOBERNADOR, JAL
  clases de opción: {'candidatura': 98228, 'voto_de_lista_no_preferente': 10814, 'opcion_especial': 7588}
  116,518 filas traen el código de comuna pegado al nombre; se conserva el original y se agrega comuna_normalizada.

REG_2022_CONGRESO (2022) · corporaciones: CAMARA, SENADO
  clases de opción: {'candidatura': 36307, 'voto_de_lista_no_preferente': 8256, 'opcion_especial': 3644}
  44,078 filas traen el código de comuna pegado al nombre; se conserva el original y se agrega comuna_normalizada.
  ATENCIÓN: 56 códigos de puesto para 61 puestos reales. Hay que agrupar por id_puesto, nunca por cod_puesto.


### 8.1. Identificación de puestos

En Congreso 2022, varios puestos rurales comparten el código `00`. En 2019 y 2023, los códigos pueden ser alfanuméricos (`A1`, `A5`, `B2`).

`id_puesto` combina zona, código y nombre normalizado. Las agregaciones territoriales usan este identificador, no `cod_puesto`.

In [9]:
if votos:
    for idf, d in votos.items():
        chocan = (d.groupby(["cod_zona", "cod_puesto"])["puesto"]
                  .nunique().rename("puestos_distintos").reset_index())
        chocan = chocan[chocan["puestos_distintos"] > 1]
        if len(chocan):
            print(f"{idf}: códigos de puesto compartidos por varios puestos")
            display(chocan)
            detalle = d[d["cod_zona"].isin(chocan["cod_zona"])
                        & d["cod_puesto"].isin(chocan["cod_puesto"])][
                ["cod_zona", "cod_puesto", "puesto", "id_puesto"]].drop_duplicates()
            escribir_csv(detalle, AUDITORIA / f"codigos_puesto_compartidos_{idf.lower()}.csv")
        else:
            print(f"{idf}: cada código de puesto identifica un solo puesto.")

REG_2023_TERR: cada código de puesto identifica un solo puesto.
REG_2019_TERR: cada código de puesto identifica un solo puesto.
REG_2022_CONGRESO: códigos de puesto compartidos por varios puestos


,cod_zona,cod_puesto,puestos_distintos
51,99,00,6


## 9. Salidas y controles

Cada corporación se guarda en una tabla separada. No se combinan elecciones distintas.

La etiqueta de corporación se toma del archivo, ya que un mismo código puede representar elecciones distintas.

In [10]:
DESTINO = PROCESADOS / "votos"
escritos, controles_todos = [], []

for idf, d in votos.items():
    anio = d["anio"].iloc[0]
    familia = d["familia"].iloc[0]
    for corp, trozo in d.groupby("corporacion", sort=True):
        orden = [c for c in (mmv.LLAVE_MESA_CSV + ["cod_corporacion", "cod_partido",
                                                   "cod_candidato"]) if c in trozo.columns]
        trozo = trozo.sort_values(orden).reset_index(drop=True)
        carpeta = DESTINO / familia / str(anio)
        carpeta.mkdir(parents=True, exist_ok=True)
        base = f"{idf.lower()}_{mmv.canon(corp).lower().replace(' ', '_')}"
        huella = mmv.sha256_bytes(trozo.to_csv(index=False, lineterminator="\n",
                                               float_format="%.6f").encode("utf-8"))
        try:
            destino = carpeta / f"{base}.parquet"
            trozo.to_parquet(destino, index=False)
            formato_salida = "parquet"
        except Exception:  # noqa: BLE001  (sin pyarrow instalado)
            destino = carpeta / f"{base}.csv"
            escribir_csv(trozo, destino)
            formato_salida = "csv"
        escritos.append({
            "id_fuente": idf, "anio": anio, "familia": familia, "corporacion": corp,
            "archivo": rel(destino), "formato": formato_salida, "filas": len(trozo),
            "puestos": int(trozo[mmv.LLAVE_GEO_CSV].drop_duplicates().shape[0]),
            "mesas": int(trozo[mmv.LLAVE_MESA_CSV].drop_duplicates().shape[0]),
            "partidos": int(trozo["cod_partido"].nunique()),
            "candidaturas_con_voto": int(
                trozo[trozo["clase_opcion"] == "candidatura"]["candidato"].nunique()),
            "votos": int(trozo["votos"].sum()),
            "sha256_csv_equivalente": huella,
        })
        controles_todos.append(mmv.controles_csv(trozo, f"{idf}·{corp}"))

tabla_escritos = pd.DataFrame(escritos)
if len(tabla_escritos):
    display(tabla_escritos[["anio", "corporacion", "filas", "puestos", "mesas", "partidos",
                            "candidaturas_con_voto", "votos", "formato"]])
    escribir_csv(tabla_escritos, AUDITORIA / "manifiesto_salidas.csv")
    print(f"\n{len(tabla_escritos)} tablas escritas en {rel(DESTINO)}")
    print(f"Total de votos extraídos: {tabla_escritos['votos'].sum():,}".replace(",", "."))
else:
    print("No se escribió ninguna tabla.")

,anio,corporacion,filas,puestos,mesas,partidos,candidaturas_con_voto,votos,formato
0,2023,ALCALDE,6109,67,593,13,12,132133,parquet
1,2023,ASAMBLEA,35191,67,593,17,184,128983,parquet
2,2023,CONCEJO,61400,67,593,26,359,131961,parquet
3,2023,GOBERNADOR,6369,67,593,10,9,129197,parquet
4,2023,JAL,29073,66,592,21,509,127484,parquet
5,2019,ALCALDE,4418,63,525,7,6,121980,parquet
6,2019,ASAMBLEA,33473,63,525,15,167,119968,parquet
7,2019,CONCEJO,51457,63,525,19,275,121765,parquet
8,2019,GOBERNADOR,4524,63,525,7,6,120891,parquet
9,2019,JAL,22758,62,534,13,324,116070,parquet



12 tablas escritas en datos/procesados/votos
Total de votos extraídos: 1.452.467


In [11]:
if controles_todos:
    controles = pd.concat(controles_todos, ignore_index=True)
    escribir_csv(controles.astype(str), AUDITORIA / "controles.csv")
    resumen_ctl = (controles.groupby(["control", "criticidad"])["pasa"]
                   .agg(pasan=lambda s: int((s == True).sum()),  # noqa: E712
                        fallan=lambda s: int((s == False).sum()),  # noqa: E712
                        no_evaluables=lambda s: int(s.isna().sum()))
                   .reset_index())
    display(resumen_ctl)
    fallidos = controles[controles["pasa"] == False]  # noqa: E712
    if len(fallidos):
        print("\nControles que no pasan:")
        display(fallidos[["tabla", "control", "valor", "esperado", "criticidad"]])
    bloqueos = sorted(set(fallidos[fallidos["criticidad"] == "Alta"]["control"]))
    alertas = sorted(set(fallidos[fallidos["criticidad"] != "Alta"]["control"]))
    no_evaluables = sorted(set(controles[controles["pasa"].isna()]["control"]))
    if no_evaluables:
        print("\nControles que no se pueden evaluar con estas fuentes:")
        for c in no_evaluables:
            print(f"  {c}")
else:
    controles, bloqueos, alertas, no_evaluables = pd.DataFrame(), [], [], []

,control,criticidad,pasan,fallan,no_evaluables
0,el código de puesto identifica un solo puesto,Media,10,2,0
1,llave puesto/mesa/corporación/partido/candidat...,Alta,12,0,0
2,mesas distintas,Baja,12,0,0
3,puestos distintos,Baja,12,0,0
4,toda fila queda clasificada,Alta,12,0,0
5,toda fila tiene corporación,Alta,12,0,0
6,votos no negativos,Alta,12,0,0
7,votos no superan el potencial de sufragantes,Alta,0,0,12



Controles que no pasan:


,tabla,control,valor,esperado,criticidad
81,REG_2022_CONGRESO·CAMARA,el código de puesto identifica un solo puesto,56 códigos para 61 puestos,"mismo número; si no, agrupar por id_puesto",Media
89,REG_2022_CONGRESO·SENADO,el código de puesto identifica un solo puesto,56 códigos para 61 puestos,"mismo número; si no, agrupar por id_puesto",Media



Controles que no se pueden evaluar con estas fuentes:
  votos no superan el potencial de sufragantes


### 9.1. Resumen por corporación y año

La tabla resume votos, mesas y candidaturas por corporación y año.

In [12]:
if votos:
    todo = pd.concat(votos.values(), ignore_index=True)
    panorama = (todo.groupby(["anio", "corporacion"])
                .agg(votos=("votos", "sum"),
                     mesas=("mesa", lambda s: 0),
                     filas=("votos", "size"),
                     partidos=("cod_partido", "nunique"))
                .reset_index())
    # Las mesas y los puestos hay que contarlos por combinación, no con nunique.
    claves = (todo.groupby(["anio", "corporacion"])
              .apply(lambda g: pd.Series({
                  "mesas": g[mmv.LLAVE_MESA_CSV].drop_duplicates().shape[0],
                  "puestos": g[mmv.LLAVE_GEO_CSV].drop_duplicates().shape[0],
                  "candidaturas": g[g["clase_opcion"] == "candidatura"]["candidato"].nunique(),
                  "votos_en_blanco_nulos_no_marcados":
                      int(g[g["clase_opcion"] == "opcion_especial"]["votos"].sum()),
              }), include_groups=False)
              .reset_index())
    panorama = panorama.drop(columns=["mesas"]).merge(claves, on=["anio", "corporacion"])
    panorama["pct_blanco_nulo_no_marcado"] = (
        panorama["votos_en_blanco_nulos_no_marcados"] / panorama["votos"] * 100).round(1)
    display(panorama[["anio", "corporacion", "votos", "puestos", "mesas", "partidos",
                      "candidaturas", "votos_en_blanco_nulos_no_marcados",
                      "pct_blanco_nulo_no_marcado"]])
    escribir_csv(panorama, PROCESADOS / "panorama_por_corporacion.csv")
    print("\nNivel de evidencia: dato descriptivo, tomado directamente de los archivos "
          "oficiales.\nQué NO dice: nada sobre participación ni abstención. Para eso hace "
          "falta el censo\nelectoral por puesto, que estos archivos no incluyen (ver la "
          "sección siguiente).")
else:
    panorama = pd.DataFrame()

,anio,corporacion,votos,puestos,mesas,partidos,candidaturas,votos_en_blanco_nulos_no_marcados,pct_blanco_nulo_no_marcado
0,2019,ALCALDE,121980,63,525,7,6,8301,6.8
1,2019,ASAMBLEA,119968,63,525,15,167,42483,35.4
2,2019,CONCEJO,121765,63,525,19,275,10276,8.4
3,2019,GOBERNADOR,120891,63,525,7,6,21259,17.6
4,2019,JAL,116070,62,534,13,324,44315,38.2
5,2022,CAMARA,100605,61,547,64,98,13786,13.7
6,2022,SENADO,101430,61,546,26,635,12860,12.7
7,2023,ALCALDE,132133,67,593,13,12,8750,6.6
8,2023,ASAMBLEA,128983,67,593,17,184,40288,31.2
9,2023,CONCEJO,131961,67,593,26,359,12107,9.2



Nivel de evidencia: dato descriptivo, tomado directamente de los archivos oficiales.
Qué NO dice: nada sobre participación ni abstención. Para eso hace falta el censo
electoral por puesto, que estos archivos no incluyen (ver la sección siguiente).


## 10. Fuentes pendientes

El formato consolidado no incluye los insumos necesarios para medir participación, abstención y candidaturas inscritas. Se registran los faltantes y los análisis que dependen de ellos.

In [13]:
PENDIENTES = [
    {"id_fuente": "REG_CENSO_HIST",
     "titulo": "Censo electoral histórico por puesto de votación",
     "bloquea": "Participación y abstención (objetivo 3.5). El censo es el denominador: sin "
                "él no hay tasa de participación, ni por municipio ni por puesto. También "
                "impide el control de que los votos no superen el potencial de sufragantes.",
     "portal": "https://www.registraduria.gov.co/-Consulta-los-censos-historicos-utilizados-"
               "para-los-procesos-electorales-desde-.html",
     "instruccion": "Exportar el censo de 2019, 2022 y 2023 para Barrancabermeja y guardarlo "
                    "en datos/brutos/registraduria/divipole/ con el año en el nombre. Debe "
                    "venir por puesto, no solo el total del municipio."},
    {"id_fuente": "GEO_COMUNAS",
     "titulo": "Límites de comunas, barrios y corregimientos de Barrancabermeja",
     "bloquea": "Todo el análisis espacial (objetivo 3.11): Moran's I, LISA y Getis-Ord "
                "necesitan geometrías, y los mapas por comuna también. Los datos ya traen "
                "código y nombre de comuna, así que solo falta la geometría.",
     "portal": "Geoportal del DANE, IGAC o Secretaría de Planeación del Distrito",
     "instruccion": "Conseguir el shapefile o GeoJSON y guardarlo en datos/geograficos/. "
                    "Debe traer un campo que empareje con cod_comuna; si no, habrá que "
                    "construir la tabla de equivalencia a mano y documentarla."},
    {"id_fuente": "CNE_CUENTAS_CLARAS",
     "titulo": "Ingresos y gastos reportados de las campañas",
     "bloquea": "El análisis de financiación (objetivo 3.13) y su cruce con contratación "
                "pública.",
     "portal": "https://www.cnecuentasclaras.gov.co/",
     "instruccion": "Exportar los informes de Barrancabermeja por elección y corporación a "
                    "datos/brutos/cne/. Toda cifra que salga de ahí lleva escrita al lado "
                    "que lo reportado no es lo gastado, y los financiadores personas "
                    "naturales se agregan: nunca se nombran en un producto público."},
]

escribir_csv(pd.DataFrame(PENDIENTES), AUDITORIA / "pendientes_no_descargables.csv")
for f in PENDIENTES:
    print(f"\n{f['id_fuente']} — {f['titulo']}")
    print(f"  Bloquea:   {f['bloquea']}")
    print(f"  Qué hacer: {f['instruccion']}")
    print(f"  Dónde:     {f['portal']}")


REG_CENSO_HIST — Censo electoral histórico por puesto de votación
  Bloquea:   Participación y abstención (objetivo 3.5). El censo es el denominador: sin él no hay tasa de participación, ni por municipio ni por puesto. También impide el control de que los votos no superen el potencial de sufragantes.
  Qué hacer: Exportar el censo de 2019, 2022 y 2023 para Barrancabermeja y guardarlo en datos/brutos/registraduria/divipole/ con el año en el nombre. Debe venir por puesto, no solo el total del municipio.
  Dónde:     https://www.registraduria.gov.co/-Consulta-los-censos-historicos-utilizados-para-los-procesos-electorales-desde-.html

GEO_COMUNAS — Límites de comunas, barrios y corregimientos de Barrancabermeja
  Bloquea:   Todo el análisis espacial (objetivo 3.11): Moran's I, LISA y Getis-Ord necesitan geometrías, y los mapas por comuna también. Los datos ya traen código y nombre de comuna, así que solo falta la geometría.
  Qué hacer: Conseguir el shapefile o GeoJSON y guardarlo en dato

### 10.1. Alcance del análisis

In [14]:
print("YA SE PUEDE RESPONDER con lo que hay en datos/procesados/:")
for linea in [
    "Cuántos votos obtuvo cada candidatura, en cada mesa y en cada puesto.",
    "Cómo se distribuyó el voto por zona, puesto y comuna.",
    "Cuántos partidos y cuántas candidaturas compitieron por cada corporación.",
    "Cuánto pesaron el voto en blanco, el nulo y el no marcado.",
    "Cuánto voto fue de lista y cuánto preferente por candidato.",
    "Comparación entre 2019 y 2023 para Alcaldía, Concejo, Asamblea, Gobernación y JAL.",
    "Comparación del comportamiento local en Congreso 2022 frente a lo territorial.",
]:
    print(f"  · {linea}")

print("\nTODAVÍA NO SE PUEDE RESPONDER:")
for linea in [
    "Participación ni abstención: falta el censo electoral por puesto.",
    "Quién ganó la curul: la asignación por cifra repartidora se calcula en la etapa "
    "siguiente, a partir de estas tablas.",
    "Nada espacial con rigor estadístico: faltan las geometrías de comuna.",
    "Nada de financiación: falta Cuentas Claras.",
    "Trayectorias entre elecciones: requiere el enlace de nombres entre años, que es una "
    "hipótesis y se construye con su propia sensibilidad, no aquí.",
]:
    print(f"  · {linea}")

YA SE PUEDE RESPONDER con lo que hay en datos/procesados/:
  · Cuántos votos obtuvo cada candidatura, en cada mesa y en cada puesto.
  · Cómo se distribuyó el voto por zona, puesto y comuna.
  · Cuántos partidos y cuántas candidaturas compitieron por cada corporación.
  · Cuánto pesaron el voto en blanco, el nulo y el no marcado.
  · Cuánto voto fue de lista y cuánto preferente por candidato.
  · Comparación entre 2019 y 2023 para Alcaldía, Concejo, Asamblea, Gobernación y JAL.
  · Comparación del comportamiento local en Congreso 2022 frente a lo territorial.

TODAVÍA NO SE PUEDE RESPONDER:
  · Participación ni abstención: falta el censo electoral por puesto.
  · Quién ganó la curul: la asignación por cifra repartidora se calcula en la etapa siguiente, a partir de estas tablas.
  · Nada espacial con rigor estadístico: faltan las geometrías de comuna.
  · Nada de financiación: falta Cuentas Claras.
  · Trayectorias entre elecciones: requiere el enlace de nombres entre años, que es una h

## 11. Estado del corpus

In [15]:
estructuras_pendientes = sorted(p.name for p in AUDITORIA.glob("estructura_pendiente_*.json"))
corporaciones = sorted(set(tabla_escritos["corporacion"])) if len(tabla_escritos) else []
anios = sorted(set(tabla_escritos["anio"].astype(str))) if len(tabla_escritos) else []
obligatorias_ok = all(a["estado"] in OK
                      for a, f in zip(adquisiciones, FUENTES) if f["obligatoria"])

if bloqueos:
    dictamen = "BLOQUEADO"
elif not obligatorias_ok:
    dictamen = "PENDIENTE_DESCARGA"
elif not len(tabla_escritos):
    dictamen = "PENDIENTE_EXTRACCION"
elif alertas or incidencias:
    dictamen = "VALIDADO_CON_ALERTAS"
else:
    dictamen = "VALIDADO"

estado = {
    "cuaderno": "01_cierre_adquisicion",
    "version_cuaderno": VERSION_NB,
    "id_ejecucion": ID_EJECUCION,
    "generado": ahora(),
    "dictamen": dictamen,
    "municipio": MUNICIPIO,
    "departamento": DEPARTAMENTO,
    "codigo_dane": DANE_MUNICIPIO,
    "codigo_electoral_observado": sorted(
        {r["cod_municipio"] for r in resumenes}) if resumenes else [],
    "plantillas_url": PLANTILLAS,
    "adquisiciones": {a["id_fuente"]: a["estado"] for a in adquisiciones},
    "formatos_detectados": {a["id_fuente"]: a.get("formato") for a in disponibles},
    "huellas_originales": {a["id_fuente"]: a.get("sha256") for a in adquisiciones},
    "huellas_publicadas_verificadas": {h["id_fuente"]: h["estado"] for h in huellas},
    "anios_cubiertos": anios,
    "corporaciones_cubiertas": corporaciones,
    "tablas_escritas": int(len(tabla_escritos)),
    "votos_extraidos": int(tabla_escritos["votos"].sum()) if len(tabla_escritos) else 0,
    "filas_extraidas": int(tabla_escritos["filas"].sum()) if len(tabla_escritos) else 0,
    "controles_alta_fallidos": bloqueos,
    "alertas": alertas,
    "controles_no_evaluables": no_evaluables,
    "incidencias": incidencias,
    "estructuras_por_declarar": estructuras_pendientes,
    "pendientes_no_descargables": [f["id_fuente"] for f in PENDIENTES],
    "pendientes_heredados_del_00": pendientes_00,
    "listo_para_analisis_descriptivo": dictamen in {"VALIDADO", "VALIDADO_CON_ALERTAS"},
}
escribir_json(AUDITORIA / "estado_corpus.json", estado)
escribir_json(RAIZ / "docs" / "fuentes" / "estado_corpus_ultimo.json", estado)

print(f"DICTAMEN DE LA ETAPA: {dictamen}\n")
print(f"Años cubiertos:      {', '.join(anios) or 'ninguno'}")
print(f"Corporaciones:       {', '.join(corporaciones) or 'ninguna'}")
print(f"Tablas escritas:     {len(tabla_escritos)}")
if len(tabla_escritos):
    print(f"Filas extraídas:     {tabla_escritos['filas'].sum():,}".replace(",", "."))
    print(f"Votos extraídos:     {tabla_escritos['votos'].sum():,}".replace(",", "."))
if bloqueos:
    print("\nBLOQUEADO por:", ", ".join(bloqueos))
    print("No se continúa a la etapa siguiente hasta resolver esto.")
if alertas:
    print("\nAlertas (no bloquean, pero quedan escritas):")
    for a_ in alertas:
        print(f"  · {a_}")
if no_evaluables:
    print("\nControles que estas fuentes no permiten evaluar:")
    for c in no_evaluables:
        print(f"  · {c}")
print(f"\nAuditoría completa: {rel(AUDITORIA)}")

DICTAMEN DE LA ETAPA: VALIDADO_CON_ALERTAS

Años cubiertos:      2019, 2022, 2023
Corporaciones:       ALCALDE, ASAMBLEA, CAMARA, CONCEJO, GOBERNADOR, JAL, SENADO
Tablas escritas:     12
Filas extraídas:     302.979
Votos extraídos:     1.452.467

Alertas (no bloquean, pero quedan escritas):
  · el código de puesto identifica un solo puesto

Controles que estas fuentes no permiten evaluar:
  · votos no superan el potencial de sufragantes

Auditoría completa: docs/fuentes/auditorias/01_cierre_adquisicion/20261003_235629_099440
